# TARDIS_MODEL

---

> **Auteur**: Laurince AGANI    
> **Dataset**: `cleaned_dataset.csv`    
> **Objectif**: Prédire le retard moyen à l'arrivée de tous les trains (`Average delay of all trains at arrival`) en minutes, et comprendre les causes des dysfonctionnements opérationnels.

---

### Plan du notebook    
1. Importation des librairies nécessaires et configurations 
2. Chargement et inspection des données
3. Preprocessing    
    3.1 Feature Engineering     
    3.2 Définition des Targets et Séparation Train/Test     
    3.3 Construction du Pipeline sklearn
4. Entraînement des modèles 
5. Évaluation et comparaison des modèles

## 1. Importation des librairies nécessaires et configurations


In [ ]:
# Manipulation de données
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Préprocessing
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Scikit-learn: Modèles de régression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)

# Scikit-learn: Evaluation
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    RandomizedSearchCV,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Scikit-learn: Dummy
from sklearn.dummy import DummyRegressor

# Sauvegarde du modèle
import joblib
import json

# Utilitaires
import warnings

warnings.filterwarnings("ignore")

# Colonne à prédire
TARGET = "Average delay of all trains at arrival"

# SEED
SEED = 42

## 2. Chargement et inspection des données

In [2]:
# Chargement du dataset
df = pd.read_csv("cleaned_dataset.csv", sep=";", encoding="utf-8")

# Inspection initiale du dataset
print("=" * 84)
print("APERÇU DES 5 PREMIÈRES LIGNES")
print("=" * 84)
display(df.head(5))

# Forme du dataset
print("Dimensions du dataset")
print(f"    {df.shape[0]} observations")
print(f"    {df.shape[1]} features")

APERÇU DES 5 PREMIÈRES LIGNES


,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Number of trains delayed at departure,Average delay of late trains at departure,Average delay of all trains at departure,...,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)",Year,Month,Delay categories,is_delayed,Cancellation_rate
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870.0,5.0,289.0,11.247809,3.693179,...,31.092437,10.924370,15.966387,5.040000,0.840336,2018,1,average delay,1,0.005747
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.00,406.0,1.0,213.0,8.479969,4.567119,...,35.000000,16.666667,16.666667,8.333333,3.333333,2018,1,average delay,1,0.002463
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226.0,0.0,21.0,6.239683,0.286283,...,27.777778,16.666667,16.666667,5.555556,11.111111,2018,1,minor delay,0,0.000000
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,71.0,7.235211,0.980000,...,22.222222,16.666667,20.370370,5.555556,1.851852,2018,1,average delay,1,0.005906
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.00,472.0,4.0,224.0,6.784673,3.229701,...,45.614035,18.750000,15.789474,1.754386,1.754386,2018,1,minor delay,0,0.008475


Dimensions du dataset
    11487 observations
    28 features


In [3]:
# Informations sur le dataset
print("=" * 84)
print("INFORMATIONS SUR LE DATASET")
print("=" * 84)
df.info()

INFORMATIONS SUR LE DATASET
<class 'pandas.DataFrame'>
RangeIndex: 11487 entries, 0 to 11486
Data columns (total 28 columns):
 #   Column                                                                         Non-Null Count  Dtype  
---  ------                                                                         --------------  -----  
 0   Date                                                                           11487 non-null  str    
 1   Service                                                                        11487 non-null  str    
 2   Departure station                                                              11487 non-null  str    
 3   Arrival station                                                                11487 non-null  str    
 4   Average journey time                                                           11487 non-null  float64
 5   Number of scheduled trains                                                     11487 non-null  float64
 6   Numbe

In [4]:
# Statistiques descriptives du dataset
print("=" * 84)
print("STATISTIQUES DESCRIPTIVES DU DATASET")
print("=" * 84)
df.describe().round(2)

STATISTIQUES DESCRIPTIVES DU DATASET


,Average journey time,Number of scheduled trains,Number of cancelled trains,Number of trains delayed at departure,Average delay of late trains at departure,Average delay of all trains at departure,Number of trains delayed at arrival,Average delay of late trains at arrival,Average delay of all trains at arrival,Number of trains delayed > 15min,...,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)",Year,Month,is_delayed,Cancellation_rate
count,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,...,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11477.00
mean,170.72,269.85,8.52,86.13,12.28,3.11,37.23,35.17,6.03,26.72,...,21.52,21.83,20.34,18.82,7.33,7.52,2021.48,6.51,0.55,inf
std,87.16,181.15,22.49,89.89,11.80,5.15,30.98,15.60,7.04,22.43,...,15.81,14.74,14.52,13.35,7.93,9.36,2.31,3.45,0.50,NaN
min,0.00,0.00,0.00,0.00,0.00,-229.27,0.00,-40.11,-472.64,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,2018.00,1.00,0.00,0.00
25%,101.00,152.00,0.00,22.00,6.28,1.22,15.00,25.95,3.43,11.00,...,10.71,12.16,10.53,10.19,0.00,0.00,2019.00,4.00,0.00,0.00
50%,164.00,229.00,2.00,52.00,10.38,2.33,29.00,33.47,5.35,21.00,...,19.05,20.00,18.75,17.11,5.88,5.00,2021.00,7.00,1.00,0.01
75%,222.00,351.00,7.00,125.00,15.57,3.89,50.00,42.25,8.00,36.00,...,29.41,29.41,28.18,25.00,10.87,11.11,2024.00,9.00,1.00,0.03
max,786.00,1100.00,297.00,1066.00,316.19,115.05,376.00,299.60,92.00,312.00,...,100.00,100.00,100.00,100.00,100.00,100.00,2025.00,12.00,1.00,inf


In [5]:
# Analyse des valeurs manquantes

# Afficher les colonnes avec des valeurs manquantes
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() * 100) / len(df)

missing_df = pd.DataFrame(
    {"Valeurs manquantes": missing, "Pourcentage (%)": missing_pct.round(2)}
).sort_values("Pourcentage (%)", ascending=False)

missing_with_nan = missing_df[missing_df["Valeurs manquantes"] > 0]

print("=" * 84)
print("VALEURS MANQUANTES PAR COLONNE")
print("=" * 84)
if len(missing_with_nan) == 0:
    print("Aucune valeur manquante détectée !")
else:
    display(missing_with_nan)
    print(f"{len(missing_with_nan)} colonnes contiennent des valeurs manquantes.")

VALEURS MANQUANTES PAR COLONNE


,Valeurs manquantes,Pourcentage (%)
Cancellation_rate,10,0.09


1 colonnes contiennent des valeurs manquantes.


In [6]:
# Distribution de la TARGET
print("=" * 84)
print(f"STATISTIQUES DE LA TARGET '{TARGET}'")
print("=" * 84)

print(f"Moyenne     : {df[TARGET].mean():.2f} minutes")
print(f"Médiane     : {df[TARGET].median():.2f} minutes")
print(f"Ecart-Type  : {df[TARGET].std():.2f} minutes")
print(f"Min         : {df[TARGET].min():.2f} minutes")
print(f"Max         : {df[TARGET].max():.2f} minutes")
print(f"Asymétrie   : {df[TARGET].skew():.3f}")

STATISTIQUES DE LA TARGET 'Average delay of all trains at arrival'
Moyenne     : 6.03 minutes
Médiane     : 5.35 minutes
Ecart-Type  : 7.04 minutes
Min         : -472.64 minutes
Max         : 92.00 minutes
Asymétrie   : -31.896


## 3. Préprocessing

Le **préprocessing** est l'ensemble des transformations appliquées aux données brutes avant de les fournir à un algorithme de machine learning. 

Notre préprocessing sera principalement constitué d'une identification du type de chaque colonne, d'un encodage catégoriel (`OneHotEncoder`), d'une normalisation (`StandardScaler`) et d'une imputation (`SimpleImputer`). Afin d'enchaîner correctement toutes ces transformations, nous utiliserons un pipeline (`Pipeline`).

In [7]:
# Identification des colonnes par type

numeric_cols_all = df.select_dtypes(include=[np.number]).columns
df[numeric_cols_all] = df[numeric_cols_all].replace([np.inf, -np.inf], np.nan)

# Colonnes catégorielles
CATEGORICAL_COLS = [
    "Service",
    "Departure station",
    "Arrival station",
    "Delay categories",
]

# Colonnes temporelles
TEMPORAL_COLS = ["Date"]

# Colonnes exclues du préprocessing
EXCLUDE_COLS = ["is_delayed", "Date", TARGET]

# Colonnes numériques
NUMERICAL_COLS = [
    col
    for col in df.select_dtypes(include=[np.number]).columns
    if col not in EXCLUDE_COLS
]

print("=" * 84)
print("CLASSIFICATION DES COLONNES")
print("=" * 84)

print(f"\nColonnes catégorielles ({len(CATEGORICAL_COLS)}):")
for c in CATEGORICAL_COLS:
    print(f"    - {c} -> {df[c].nunique()} valeurs uniques")

print(f"\nColonnes numériques ({len(NUMERICAL_COLS)}):")
for c in NUMERICAL_COLS:
    print(f"    - {c}")

print(f"\nColonnes temporelles ({len(TEMPORAL_COLS)}):")
for c in TEMPORAL_COLS:
    print(f"    - {c}")

print(f"\nColonnes exclues ({len(EXCLUDE_COLS)}):")
for c in EXCLUDE_COLS:
    print(f"    - {c}")

CLASSIFICATION DES COLONNES

Colonnes catégorielles (4):
    - Service -> 2 valeurs uniques
    - Departure station -> 131 valeurs uniques
    - Arrival station -> 116 valeurs uniques
    - Delay categories -> 5 valeurs uniques

Colonnes numériques (21):
    - Average journey time
    - Number of scheduled trains
    - Number of cancelled trains
    - Number of trains delayed at departure
    - Average delay of late trains at departure
    - Average delay of all trains at departure
    - Number of trains delayed at arrival
    - Average delay of late trains at arrival
    - Number of trains delayed > 15min
    - Average delay of trains > 15min (if competing with flights)
    - Number of trains delayed > 30min
    - Number of trains delayed > 60min
    - Pct delay due to external causes
    - Pct delay due to infrastructure
    - Pct delay due to traffic management
    - Pct delay due to rolling stock
    - Pct delay due to station management and equipment reuse
    - Pct delay due to p

### 3.1. Feature Engineering

Le **feature engineering** est l'art de créer de nouvelles variables à partir des variables existantes pour aider le modèle à mieux apprendre.

In [8]:
# Création de nouvelles features

df_feat = df.copy()

# Indicateurs saisonniers
month_col = df_feat["Month"]

# On encode la saison de manière cyclique pour capturer la continuité temporelle
df_feat["month_sin"] = np.sin(2 * np.pi * month_col / 12)
df_feat["month_cos"] = np.cos(2 * np.pi * month_col / 12)

# Indicateur de saison (1 = hiver, 2 = printemps, 3 = été, 4 = automne)
df_feat["season"] = pd.cut(
    month_col, bins=[0, 3, 6, 9, 12], labels=[1, 2, 3, 4], include_lowest=True
).astype(float)

# Indicateur de pic
df_feat["is_peak_month"] = month_col.isin([7, 8, 12, 1]).astype(int)

print(
    "Features saisonnières créées ('month_sin', 'month_cos', 'season', 'is_peak_month')"
)

print("\n" + "=" * 84)
print("FEATURE ENGINEERING TERMINÉ")
print(f"   Nouvelles features : {df_feat.shape[1] - df.shape[1]}")
print(f"   Dataset étendu : {df_feat.shape[0]} lignes X {df_feat.shape[1]} colonnes")
print("=" * 84)

Features saisonnières créées ('month_sin', 'month_cos', 'season', 'is_peak_month')

FEATURE ENGINEERING TERMINÉ
   Nouvelles features : 4
   Dataset étendu : 11487 lignes X 32 colonnes


### 3.2 Définition des Targets et Séparation Train/Test

Après compréhension du sujet, nous avons choisi la variable `Average delay of all trains at arrival` comme variable cible. Il s'agira donc de prédire la valeur de cette varibale à partir des autres caractéristiques connues.

Pour la séparation du dataset, nous avons choisi d'utiliser **80%** des données pour permettre au modèle d'apprendre (`train_set`) et d'évaluer le modèle sur les **20%** restants (`test_set`).

On garantit aussi la reproductivité du modèle en choisissant un seed de **42**.

In [ ]:
# Définition de la TARGET et des FEATURES

print("=" * 84)
print("DÉFINITION DES TARGETS ET SÉPARATION TRAIN/TEST")
print("=" * 84)

TARGET = "Average delay of all trains at arrival"

# Colonnes à exclure des features
EXCLUDE_FROM_FEATURES = [
    TARGET,
    "is_delayed",
    "Date",
    "Cancellation_rate",
    # "Pct delay due to external causes",
    # "Pct delay due to infrastructure",
    # "Pct delay due to traffic management",
    # "Pct delay due to rolling stock",
    # "Pct delay due to station management and equipment reuse",
    # "Pct delay due to passenger handling (crowding, disabled persons, connections)",
    "Number of trains delayed at departure",
    # "Number of cancelled trains",
    "Number of trains delayed > 60min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 15min",
    "Number of trains delayed at arrival",
    "Average delay of late trains at departure",
    "Average delay of trains > 15min (if competing with flights)",
    "Average delay of late trains at arrival",
]

# Colonnes à utiliser comme features
FEATURE_COLS = [c for c in df_feat.columns if c not in EXCLUDE_FROM_FEATURES]

print(f"Features utilisées pour la régression : {len(FEATURE_COLS)}")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"   {i:2d}. {col}")

# Séparation X / y

# X = matrice de features (inputs)
# y = vecteur target (output à prédire)

X = df_feat[FEATURE_COLS].copy()
y = df_feat[TARGET].copy()

# Supprimer les lignes où la target est NaN
mask = y.notna()
X = X[mask]
y = y[mask]

print(f"\nX (features) : {X.shape}")
print(f"y (target)    : {y.shape}")

# Train / Test Split

# 80% d'entraînement, 20% de test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, shuffle=True
)

print(f"\nRépartition Train/Test :")
print(
    f"   Train : {X_train.shape[0]:,} observations ({((X_train.shape[0] / len(X)) * 100):.1f}%)"
)
print(
    f"   Test  : {X_test.shape[0]:,} observations ({((X_test.shape[0] / len(X)) * 100):.1f}%)"
)

# Vérification de la distribution de y_train vs y_test
print(f"\nVérification de la distribution :")
print(f"   Moyenne y_train : {y_train.mean():.3f} min")
print(f"   Moyenne y_test  : {y_test.mean():.3f} min")

DÉFINITION DES TARGETS ET SÉPARATION TRAIN/TEST
Features utilisées pour la régression : 19
    1. Service
    2. Departure station
    3. Arrival station
    4. Average journey time
    5. Number of scheduled trains
    6. Number of cancelled trains
    7. Pct delay due to external causes
    8. Pct delay due to infrastructure
    9. Pct delay due to traffic management
   10. Pct delay due to rolling stock
   11. Pct delay due to station management and equipment reuse
   12. Pct delay due to passenger handling (crowding, disabled persons, connections)
   13. Year
   14. Month
   15. Delay categories
   16. month_sin
   17. month_cos
   18. season
   19. is_peak_month

X (features) : (11487, 19)
y (target)    : (11487,)

Répartition Train/Test :
   Train : 9,189 observations (80.0%)
   Test  : 2,298 observations (20.0%)

Vérification de la distribution :
   Moyenne y_train : 6.000 min
   Moyenne y_test  : 6.159 min


### 3.3 Construction du Pipeline Sklearn

In [ ]:
# Construction du Pipeline sklearn
print("=" * 84)
print("CONSTRUCTION DU PIPELINE SKLEARN")
print("=" * 84)

# Colonnes catégorielles présentes dans X
cat_in_X = [c for c in CATEGORICAL_COLS if c in X_train.columns]

# Colonnes numériques présentes dans X (toutes sauf catégorielles)
num_in_X = [c for c in X_train.columns if c not in cat_in_X]

print(f"Colonnes catégorielles dans X : {cat_in_X}")
print(f"Colonnes numériques dans X    : {len(num_in_X)} colonnes")

# Pipeline numérique (Imputation médiane+ Normalisation)
numeric_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Pipeline catégorielle (Imputation mode + Encodage binaire)
categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

# ColumnTransformer (appliquer le bon Pipeline à chaque colonne)
preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, num_in_X),
        ("categorical", categorical_pipeline, cat_in_X),
    ],
    remainder="drop",
)

print("\nPreprocessor créé avec succès !")
print("   -> Pipeline numérique  : Imputation médiane + StandardScaler")
print("   -> Pipeline catégoriel : Imputation mode + OneHotEncoder")
print("   -> ColumnTransformer   : Applique les bons traitements par type de colonne")

CONSTRUCTION DU PIPELINE SKLEARN
Colonnes catégorielles dans X : ['Service', 'Departure station', 'Arrival station', 'Delay categories']
Colonnes numériques dans X    : 15 colonnes

Preprocessor créé avec succès !
   -> Pipeline numérique  : Imputation médiane + StandardScaler
   -> Pipeline catégoriel : Imputation mode + OneHotEncoder
   -> ColumnTransformer   : Applique les bons traitements par type de colonne


## 4. Entraînement des modèles

Nous allons entraîner **6 modèles** de régression.

In [11]:
# MODÈLE 0 — BASELINE: Prédire toujours la moyenne (DummyRegressor)

baseline = Pipeline(
    [("preprocessor", preprocessor), ("model", DummyRegressor(strategy="mean"))]
)

baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
r2_baseline = r2_score(y_test, y_pred_baseline)

print(f"BASELINE (prédire la moyenne) :")
print(f"   RMSE : {rmse_baseline:.3f} min  — Racine de l'erreur quadratique moyenne")
print(f"   MAE  : {mae_baseline:.3f} min  — Erreur absolue moyenne")
print(
    f"   R²   : {r2_baseline:.4f}      — Doit être 0.0 par définition pour DummyRegressor"
)

BASELINE (prédire la moyenne) :
   RMSE : 5.714 min  — Racine de l'erreur quadratique moyenne
   MAE  : 2.965 min  — Erreur absolue moyenne
   R²   : -0.0008      — Doit être 0.0 par définition pour DummyRegressor


In [12]:
# MODÈLE 1 - RÉGRESSION LINÉAIRE

pipe_lr = Pipeline(
    [("preprocessor", preprocessor), ("model", LinearRegression(fit_intercept=True))]
)

pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"RÉGRESSION LINÉAIRE :")
print(f"   RMSE : {rmse_lr:.3f} min")
print(f"   MAE  : {mae_lr:.3f} min")
print(f"   R²   : {r2_lr:.4f}")

RÉGRESSION LINÉAIRE :
   RMSE : 4.321 min
   MAE  : 1.934 min
   R²   : 0.4277


In [13]:
# MODÈLE 2 - RANDOM FOREST REGRESSOR

pipe_rf = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,  # Nombre d'arbres. Plus = meilleur jusqu'à un plateau
                max_depth=15,  # Profondeur max de chaque arbre. Limite l'overfitting.
                min_samples_split=5,  # Nombre minimum d'exemples pour splitter un nœud
                min_samples_leaf=2,  # Nombre minimum d'exemples dans une feuille
                max_features="sqrt",  # Nombre de features à considérer à chaque split
                n_jobs=2,
                random_state=SEED,
            ),
        ),
    ]
)

print("Entraînement RandomForest... (peut prendre quelques secondes)")
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"RANDOM FOREST :")
print(f"   RMSE : {rmse_rf:.3f} min")
print(f"   MAE  : {mae_rf:.3f} min")
print(f"   R²   : {r2_rf:.4f}")

Entraînement RandomForest... (peut prendre quelques secondes)
RANDOM FOREST :
   RMSE : 4.191 min
   MAE  : 1.624 min
   R²   : 0.4616


In [14]:
# MODÈLE 3 - GRADIENT BOOSTING REGRESSOR

pipe_gb = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=200,  # Nombre d'itérations de boosting
                learning_rate=0.01,  # Taux d'apprentissage. Petit = plus robuste mais plus lent
                max_depth=5,  # Arbres plus petits (stumps) pour le boosting
                subsample=0.8,  # Fraction des données à utiliser par arbre
                min_samples_split=10,  # Nœuds minimum pour splitter
                random_state=SEED,
            ),
        ),
    ]
)

print("Entraînement GradientBoosting...")
pipe_gb.fit(X_train, y_train)
y_pred_gb = pipe_gb.predict(X_test)

rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
mae_gb = mean_absolute_error(y_test, y_pred_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print(f"GRADIENT BOOSTING :")
print(f"   RMSE : {rmse_gb:.3f} min")
print(f"   MAE  : {mae_gb:.3f} min")
print(f"   R²   : {r2_gb:.4f}")

Entraînement GradientBoosting...
GRADIENT BOOSTING :
   RMSE : 4.315 min
   MAE  : 1.640 min
   R²   : 0.4293


In [15]:
# MODÈLE 4 - EXTRA TREES REGRESSOR

pipe_et = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "model",
            ExtraTreesRegressor(
                n_estimators=300,
                max_depth=15,
                min_samples_split=5,
                min_samples_leaf=2,
                max_features="sqrt",
                n_jobs=2,
                random_state=SEED,
            ),
        ),
    ]
)

print("Entraînement ExtraTrees...")
pipe_et.fit(X_train, y_train)
y_pred_et = pipe_et.predict(X_test)

rmse_et = np.sqrt(mean_squared_error(y_test, y_pred_et))
mae_et = mean_absolute_error(y_test, y_pred_et)
r2_et = r2_score(y_test, y_pred_et)

print(f"EXTRA TREES :")
print(f"   RMSE : {rmse_et:.3f} min")
print(f"   MAE  : {mae_et:.3f} min")
print(f"   R²   : {r2_et:.4f}")

Entraînement ExtraTrees...
EXTRA TREES :
   RMSE : 4.251 min
   MAE  : 1.662 min
   R²   : 0.4461


In [16]:
# MODÈLE 5 - HIST GRADIENT BOOSTRING REGRESSOR

pipe_hgb = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=300,  # Nombre d'itérations de boosting
                learning_rate=0.01,  # Même raisonnement que GradientBoosting
                max_depth=6,  # Profondeur des arbres
                l2_regularization=0.1,  # Régularisation L2 pour réduire l'overfitting
                min_samples_leaf=20,  # Feuilles avec minimum 20 exemples
                max_bins=255,  # Nombre de bins pour les histogrammes (max = 255)
                random_state=SEED,
            ),
        ),
    ]
)

print("Entraînement HistGradientBoosting...")
pipe_hgb.fit(X_train, y_train)
y_pred_hgb = pipe_hgb.predict(X_test)

rmse_hgb = np.sqrt(mean_squared_error(y_test, y_pred_hgb))
mae_hgb = mean_absolute_error(y_test, y_pred_hgb)
r2_hgb = r2_score(y_test, y_pred_hgb)

print(f"HIST GRADIENT BOOSTING :")
print(f"   RMSE : {rmse_hgb:.3f} min")
print(f"   MAE  : {mae_hgb:.3f} min")
print(f"   R²   : {r2_hgb:.4f}")

Entraînement HistGradientBoosting...
HIST GRADIENT BOOSTING :
   RMSE : 4.072 min
   MAE  : 1.661 min
   R²   : 0.4919


## 5. Évaluation et Comparaison des modèles

| Métrique | Interprétation | Sensible aux outliers |
|----------|---------------|----------------------|
| **RMSE** | En minutes — pénalise les grosses erreurs | Oui |
| **MAE** | En minutes — robuste | Non |
| **R²** | Entre 0 et 1 (ou <0) — % de variance expliquée |  Partiel |

**RMSE vs MAE** : RMSE pénalise davantage les grandes erreurs (erreur² > erreur si erreur > 1). Pour les retards, les grandes erreurs sont importantes donc RMSE est notre métrique principale.

**R² = 0.80** signifie que notre modèle explique 80% de la variabilité des retards.


In [17]:
# TABLEAU COMPARATIF — Tous les modèles

results = {
    "Baseline (Moyenne)": {
        "RMSE": rmse_baseline,
        "MAE": mae_baseline,
        "R²": r2_baseline,
        "preds": y_pred_baseline,
    },
    "Régression Linéaire": {
        "RMSE": rmse_lr,
        "MAE": mae_lr,
        "R²": r2_lr,
        "preds": y_pred_lr,
    },
    "Random Forest": {"RMSE": rmse_rf, "MAE": mae_rf, "R²": r2_rf, "preds": y_pred_rf},
    "Gradient Boosting": {
        "RMSE": rmse_gb,
        "MAE": mae_gb,
        "R²": r2_gb,
        "preds": y_pred_gb,
    },
    "Extra Trees": {"RMSE": rmse_et, "MAE": mae_et, "R²": r2_et, "preds": y_pred_et},
    "HistGradientBoosting": {
        "RMSE": rmse_hgb,
        "MAE": mae_hgb,
        "R²": r2_hgb,
        "preds": y_pred_hgb,
    },
}

# Créer le DataFrame des résultats
results_df = pd.DataFrame(
    {
        name: {"RMSE": v["RMSE"], "MAE": v["MAE"], "R²": v["R²"]}
        for name, v in results.items()
    }
).T.round(4)

# Trier par RMSE croissant (meilleur en premier)
results_df = results_df.sort_values("RMSE")

print("=" * 70)
print("TABLEAU COMPARATIF DES MODÈLES (trié par RMSE)")
print("=" * 70)
display(
    results_df.style.background_gradient(subset=["RMSE"], cmap="RdYlGn_r")
    .background_gradient(subset=["MAE"], cmap="RdYlGn_r")
    .background_gradient(subset=["R²"], cmap="RdYlGn")
    .format({"RMSE": "{:.4f}", "MAE": "{:.4f}", "R²": "{:.4f}"})
)

best_model_name = results_df.index[0]
print(f"\nMeilleur modèle : {best_model_name}")
print(f"   RMSE : {results_df.loc[best_model_name, 'RMSE']:.4f} min")
print(f"   MAE  : {results_df.loc[best_model_name, 'MAE']:.4f} min")
print(f"   R²   : {results_df.loc[best_model_name, 'R²']:.4f}")

TABLEAU COMPARATIF DES MODÈLES (trié par RMSE)


,RMSE,MAE,R²
HistGradientBoosting,4.0716,1.6611,0.4919
Random Forest,4.1914,1.6237,0.4616
Extra Trees,4.2510,1.6624,0.4461
Gradient Boosting,4.3151,1.6403,0.4293
Régression Linéaire,4.3210,1.9336,0.4277
Baseline (Moyenne),5.7143,2.9646,-0.0008



Meilleur modèle : HistGradientBoosting
   RMSE : 4.0716 min
   MAE  : 1.6611 min
   R²   : 0.4919


In [18]:
# CROSS-VALIDATION — Validation croisée à 5 plis

print("Calcul de la validation croisée (5 plis) pour chaque modèle...")
print("   (Cette étape peut prendre quelques minutes)")

# Pipes à évaluer en CV
pipes_cv = {
    "Régression Linéaire": pipe_lr,
    "Random Forest": pipe_rf,
    "Gradient Boosting": pipe_gb,
    "Extra Trees": pipe_et,
    "HistGradientBoosting": pipe_hgb,
}

cv_results = {}

for name, pipe in pipes_cv.items():
    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=5,  # 5 plis
        scoring="neg_root_mean_squared_error",  # RMSE négatif
        n_jobs=-1,  # Parallélisation
    )
    cv_results[name] = {
        "mean_rmse": -scores.mean(),  # RMSE moyen (positif)
        "std_rmse": scores.std(),  # Écart-type du RMSE
        "scores": -scores,  # Tous les scores
    }
    print(f"   {name} : RMSE = {-scores.mean():.3f} ± {scores.std():.3f}")

Calcul de la validation croisée (5 plis) pour chaque modèle...
   (Cette étape peut prendre quelques minutes)
   Régression Linéaire : RMSE = 4.962 ± 3.836
   Random Forest : RMSE = 4.513 ± 4.266
   Gradient Boosting : RMSE = 5.997 ± 3.947
   Extra Trees : RMSE = 4.468 ± 4.245
   HistGradientBoosting : RMSE = 4.521 ± 4.149


## 6. Hyperparamter Tuning

Blabla

In [19]:
# RANDOMIZED SEARCH CV — Optimisation du Random Forest

# Définition de l'espace de recherche des hyperparamètres
param_dist = {
    # Nombre d'arbres : plus = plus stable, mais plus lent
    "model__n_estimators": [100, 200, 300, 400],
    # Profondeur max : contrôle l'overfitting
    "model__max_depth": [5, 8, 12, 15, 20, None],
    # Fraction de features à chaque split
    "model__max_features": ["sqrt", "log2", 0.5, 0.7],
    # Taille minimale des feuilles : régularisation
    "model__min_samples_leaf": [1, 2, 5, 10],
    # Minimum d'échantillons pour splitter un nœud interne
    "model__min_samples_split": [2, 5, 10, 20],
}

print("Espace de recherche :")
total_combinations = 4 * 6 * 4 * 4 * 4
print(f"   Nombre total de combinaisons possibles : {total_combinations:,}")
print(f"   Combinaisons testées (RandomizedSearch) : 30")
print(f"   Économie : {(1 - (30 / total_combinations)) * 100:.1f}%")

# RandomizedSearchCV avec 5-fold CV
random_search = RandomizedSearchCV(
    estimator=Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", RandomForestRegressor(n_jobs=-1, random_state=SEED)),
        ]
    ),
    param_distributions=param_dist,
    n_iter=30,  # Tester 30 combinaisons aléatoires
    cv=5,  # 5-fol30/total_combinationsd cross-validation
    scoring="neg_root_mean_squared_error",  # Minimiser RMSE
    n_jobs=-1,  # Parallélisation
    verbose=1,
    random_state=SEED,
)

print("\nLancement du RandomizedSearchCV... (peut prendre quelques minutes)")
random_search.fit(X_train, y_train)

print("\nRecherche terminée !")
print(f"\n Meilleurs hyperparamètres :")
for param, value in random_search.best_params_.items():
    param_clean = param.replace("model__", "")
    print(f"   {param_clean} = {value}")

best_score = -random_search.best_score_
print(f"\nMeilleur RMSE (CV) : {best_score:.4f} min")

# Évaluer le meilleur modèle sur le test set
y_pred_tuned = random_search.best_estimator_.predict(X_test)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print(f"\nPerformance sur Test Set (après tuning) :")
print(f"   RMSE : {rmse_tuned:.4f} min  (avant tuning : {rmse_rf:.4f})")
print(f"   MAE  : {mae_tuned:.4f} min  (avant tuning : {mae_rf:.4f})")
print(f"   R²   : {r2_tuned:.4f}      (avant tuning : {r2_rf:.4f})")
print(f"   Gain RMSE : {rmse_rf - rmse_tuned:.4f} min")

Espace de recherche :
   Nombre total de combinaisons possibles : 1,536
   Combinaisons testées (RandomizedSearch) : 30
   Économie : 98.0%

Lancement du RandomizedSearchCV... (peut prendre quelques minutes)
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Recherche terminée !

 Meilleurs hyperparamètres :
   n_estimators = 200
   min_samples_split = 5
   min_samples_leaf = 1
   max_features = 0.5
   max_depth = 20

Meilleur RMSE (CV) : 4.9212 min

Performance sur Test Set (après tuning) :
   RMSE : 3.6175 min  (avant tuning : 4.1914)
   MAE  : 1.4746 min  (avant tuning : 1.6237)
   R²   : 0.5989      (avant tuning : 0.4616)
   Gain RMSE : 0.5739 min


## 7. Sauvegarde du modèle

In [20]:
# SAUVEGARDE DU PIPELINE COMPLET

# Le modèle final est le meilleur modèle après tuning
final_pipeline = random_search.best_estimator_

# Chemin de sauvegarde
model_path = "model.joblib"

# Sauvegarder avec compression niveau 3 (bon compromis taille/vitesse)
joblib.dump(final_pipeline, model_path, compress=3)

import os

file_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Modèle sauvegardé : {model_path}")
print(f"   Taille du fichier : {file_size_mb:.2f} MB")
print(f"   Meilleurs hyperparamètres : {random_search.best_params_}")

# Vérification de la sauvegarde
loaded_pipeline = joblib.load(model_path)
y_pred_loaded = loaded_pipeline.predict(X_test)
rmse_loaded = np.sqrt(mean_squared_error(y_test, y_pred_loaded))

print(f"\nVérification du modèle chargé :")
print(f"   RMSE (chargé) : {rmse_loaded:.4f}")
print(f"   RMSE (original) : {rmse_tuned:.4f}")
print(f"   Identique : {'OUI' if abs(rmse_loaded - rmse_tuned) < 1e-6 else 'NON'}")

# Sauvegarder les métadonnées du modèle

metadata = {
    "model_type": "RandomForestRegressor (RandomizedSearchCV tuned)",
    "target": TARGET,
    "best_params": {k: str(v) for k, v in random_search.best_params_.items()},
    "test_rmse": float(rmse_tuned),
    "test_mae": float(mae_tuned),
    "test_r2": float(r2_tuned),
    "n_features": len(FEATURE_COLS),
    "feature_cols": FEATURE_COLS,
    "cat_cols": cat_in_X,
    "num_cols": num_in_X,
    "train_size": len(X_train),
    "test_size": len(X_test),
}

with open("model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\nMétadonnées sauvegardées : model_metadata.json")

# Résumé final
print("\n" + "=" * 70)
print("RÉSUMÉ FINAL DU PROJET TARDIS — MODELING")
print("=" * 70)
print(f"   Dataset         : {df.shape[0]:,} observations × {df.shape[1]} colonnes")
print(f"   Features créées : {df_feat.shape[1] - df.shape[1]} nouvelles features")
print(f"   Meilleur modèle : Random Forest (après RandomizedSearchCV)")
print(f"   RMSE Test       : {rmse_tuned:.3f} minutes")
print(f"   MAE Test        : {mae_tuned:.3f} minutes")
print(f"   R² Test         : {r2_tuned:.4f}")
print(f"   Fichier modèle  : {model_path} ({file_size_mb:.1f} MB)")
print("=" * 70)

Modèle sauvegardé : model.joblib
   Taille du fichier : 14.57 MB
   Meilleurs hyperparamètres : {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 0.5, 'model__max_depth': 20}

Vérification du modèle chargé :
   RMSE (chargé) : 3.6175
   RMSE (original) : 3.6175
   Identique : OUI

Métadonnées sauvegardées : model_metadata.json

RÉSUMÉ FINAL DU PROJET TARDIS — MODELING
   Dataset         : 11,487 observations × 28 colonnes
   Features créées : 4 nouvelles features
   Meilleur modèle : Random Forest (après RandomizedSearchCV)
   RMSE Test       : 3.617 minutes
   MAE Test        : 1.475 minutes
   R² Test         : 0.5989
   Fichier modèle  : model.joblib (14.6 MB)
